# ReceiptGuard-ML: Model 1 Training (LayoutLM NER)

This notebook trains the LayoutLM-based Named Entity Recognition model to extract
receipt fields (company, date, address, total) from the SROIE2019 dataset.

**Requirements:**
- Kaggle GPU (T4 or P100)
- SROIE2019 dataset added to notebook input (`urbikn/sroie-datasetv2`)

---

## 1. Install Dependencies

In [ ]:
!pip install -q transformers torch torchvision tqdm tensorboard scikit-learn
!pip install -q sentencepiece tiktoken pillow matplotlib seaborn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone Repository & Setup Paths

In [ ]:
import os, sys, glob
from pathlib import Path

WORKING_DIR = '/kaggle/working'
REPO_DIR = f'{WORKING_DIR}/Receipt_Guard'

# Clone the repo (contains config.yaml + all source code)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MoeenUddin01/Receipt_Guard.git {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Change into the repo directory so config.yaml is found at project root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"config.yaml exists: {os.path.exists('config.yaml')}")

# Add repo to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. Discover SROIE Dataset & Configure Paths

In [ ]:
# Auto-discover SROIE2019 dataset path on Kaggle
# Kaggle datasets can be at different nested levels depending on how they're added
SROIE_PATH = None
search_patterns = [
    '/kaggle/input/sroie-datasetv2/SROIE2019',
    '/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019',
    '/kaggle/input/sroie2019/SROIE2019',
    '/kaggle/input/sroie2019',
]

for path in search_patterns:
    if os.path.exists(path):
        # Verify it has train/test subdirs
        if os.path.exists(f'{path}/train') and os.path.exists(f'{path}/test'):
            SROIE_PATH = path
            break

# If standard patterns fail, do a recursive search
if SROIE_PATH is None:
    matches = glob.glob('/kaggle/input/**/SROIE2019', recursive=True)
    for m in matches:
        if os.path.exists(f'{m}/train') and os.path.exists(f'{m}/test'):
            SROIE_PATH = m
            break

if SROIE_PATH is None:
    # Last resort: check if train/test are directly in an input folder
    for d in glob.glob('/kaggle/input/*'):
        if os.path.exists(f'{d}/train/box') and os.path.exists(f'{d}/test/box'):
            SROIE_PATH = d
            break

if SROIE_PATH is None:
    print("❌ SROIE2019 dataset not found!")
    print("Please add 'urbikn/sroie-datasetv2' dataset to this notebook.")
    print("\nAvailable inputs:")
    !find /kaggle/input -maxdepth 4 -type d | head -30
    raise FileNotFoundError("SROIE2019 dataset not found under /kaggle/input")
else:
    print(f"✅ SROIE2019 found at: {SROIE_PATH}")
    for split in ['train', 'test']:
        sp = f'{SROIE_PATH}/{split}'
        if os.path.exists(sp):
            img_count = len(os.listdir(f'{sp}/img')) if os.path.exists(f'{sp}/img') else 0
            box_count = len(os.listdir(f'{sp}/box')) if os.path.exists(f'{sp}/box') else 0
            ent_count = len(os.listdir(f'{sp}/entities')) if os.path.exists(f'{sp}/entities') else 0
            print(f"  {split}: {img_count} images, {box_count} box files, {ent_count} entity files")

In [ ]:
# Now import and configure the project
from src.config import override_config, CFG

# Override all paths for Kaggle environment
override_config({
    # Data paths – point to the Kaggle input dataset
    'paths.raw_data_dir': SROIE_PATH,
    'data.raw_data_path': SROIE_PATH,
    'data.processed_data_path': f'{WORKING_DIR}/processed',
    'paths.processed_data_dir': f'{WORKING_DIR}/processed',

    # Model – use HuggingFace hub (Kaggle has internet access)
    'model.model_path': 'microsoft/layoutlm-base-uncased',
    'paths.model_dir': 'microsoft/layoutlm-base-uncased',

    # Output paths
    'training.output_dir': f'{WORKING_DIR}/checkpoints',
    'paths.artifacts_dir': f'{WORKING_DIR}/artifacts',
    'paths.checkpoints_dir': f'{WORKING_DIR}/checkpoints',
    'paths.evaluation_dir': f'{WORKING_DIR}/evaluation',
    'paths.logs_dir': f'{WORKING_DIR}/logs',

    # Training hyperparameters
    'training.num_epochs': 15,
    'training.batch_size': 8,
    'training.learning_rate': 5e-5,
})

# Verify config
print("\n📋 Config Summary:")
print(f"  Data path:   {CFG.data.raw_data_path}")
print(f"  Model:       {CFG.model.model_path}")
print(f"  Output dir:  {CFG.training.output_dir}")
print(f"  Epochs:      {CFG.training.num_epochs}")
print(f"  Batch size:  {CFG.training.batch_size}")
print(f"  LR:          {CFG.training.learning_rate}")

## 4. Run Preprocessing Pipeline

In [ ]:
from src.pipelines.preprocessing_pipeline import run_preprocessing_pipeline, PreprocessingConfig

preprocess_config = PreprocessingConfig(
    raw_data_path=SROIE_PATH,
    processed_data_path=f'{WORKING_DIR}/processed',
    splits=['train', 'test']
)

print("🔄 Running preprocessing pipeline...")
preprocess_summary = run_preprocessing_pipeline(preprocess_config)

print(f"\n✅ Preprocessing complete!")
print(f"  Total samples: {preprocess_summary['total_samples']}")
print(f"  Failed:        {preprocess_summary['failed_samples_count']}")

## 5. Verify Preprocessed Data

In [ ]:
import json

processed_dir = f'{WORKING_DIR}/processed'

try:
    with open(f'{processed_dir}/train_samples.json', 'r') as f:
        train_samples = json.load(f)
    with open(f'{processed_dir}/label_stats.json', 'r') as f:
        label_stats = json.load(f)

    print(f"✅ Train samples: {len(train_samples)}")
    print(f"\nLabel distribution:")
    for label, count in sorted(label_stats['label_counts'].items()):
        pct = (count / label_stats['total_labels']) * 100
        print(f"  {label:<12}: {count:>6} ({pct:>5.1f}%)")

    # Spot-check first sample
    sample = train_samples[0]
    entity_labels = [l for l in sample['labels'] if l != 'O']
    print(f"\nFirst sample: {len(sample['labels'])} tokens, {len(entity_labels)} entity labels")
    if entity_labels:
        print(f"  Entity types: {set(l.split('-')[1] for l in entity_labels)}")
        print("🎉 Entity labels found – preprocessing is correct!")
    else:
        print("⚠️  No entity labels in first sample (may still be fine overall)")

except FileNotFoundError as e:
    print(f"❌ Preprocessed file not found: {e}")
    print("Check preprocessing output above for errors.")

## 6. Train Model

In [ ]:
from src.pipelines.model_training_pipeline import run_training_pipeline, TrainingConfig

# Create training config – pulls defaults from CFG (which we already overrode)
training_config = TrainingConfig(
    model_path='microsoft/layoutlm-base-uncased',
    num_labels=9,
    dropout=0.1,
    output_dir=f'{WORKING_DIR}/checkpoints',
    num_epochs=15,
    batch_size=8,
    max_length=512,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    seed=42,
    data_path=SROIE_PATH,
)

print("🚀 Starting Model 1 training...")
print(f"  Data:       {training_config.data_path}")
print(f"  Output:     {training_config.output_dir}")
print(f"  Epochs:     {training_config.num_epochs}")
print(f"  Batch size: {training_config.batch_size}")
print("=" * 60)

try:
    summary = run_training_pipeline(training_config)

    if summary['status'] == 'completed':
        print("\n" + "=" * 60)
        print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
        print("=" * 60)
        print(f"  Best eval loss:  {summary['best_eval_loss']:.4f}")
        print(f"  Best checkpoint: {summary['best_checkpoint']}")
    else:
        print(f"\n❌ Training failed: {summary.get('error', 'Unknown error')}")
        print(f"   Suggestion: {summary.get('suggestion', '')}")

except Exception as e:
    print(f"\n❌ Training error: {e}")
    import traceback
    traceback.print_exc()

## 7. Verify Checkpoint

In [ ]:
import glob

# Search for checkpoints
checkpoint_patterns = [
    f'{WORKING_DIR}/checkpoints/best_model.pt',
    f'{WORKING_DIR}/checkpoints/final_model.pt',
    f'{WORKING_DIR}/**/best_model.pt',
]

found = []
for pattern in checkpoint_patterns:
    found.extend(glob.glob(pattern, recursive=True))

if found:
    for f in found:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"✅ {f} ({size_mb:.1f} MB)")
else:
    print("❌ No checkpoint files found.")
    print("Searching everywhere under /kaggle/working:")
    !find /kaggle/working -name '*.pt' -type f 2>/dev/null

## 8. Quick Model Test

In [ ]:
try:
    import torch
    from src.model.model import ReceiptFieldExtractor
    from transformers import LayoutLMTokenizer

    # Find best checkpoint
    checkpoint_path = None
    for p in [f'{WORKING_DIR}/checkpoints/best_model.pt',
              f'{WORKING_DIR}/checkpoints/final_model.pt']:
        if os.path.exists(p):
            checkpoint_path = p
            break

    if checkpoint_path:
        print(f"🧪 Testing model from: {checkpoint_path}")

        model = ReceiptFieldExtractor.load_from_checkpoint(
            checkpoint_path,
            model_path='microsoft/layoutlm-base-uncased',
            num_labels=9
        )
        model.eval()

        tokenizer = LayoutLMTokenizer.from_pretrained('microsoft/layoutlm-base-uncased')

        # Sample receipt tokens
        sample_tokens = ["BOOK", "TA", ".K", "(TAMAN", "DAYA)", "SDN", "BHD"]

        encoding = tokenizer(
            sample_tokens,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt',
        )

        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        token_type_ids = torch.zeros_like(input_ids)
        bbox = torch.zeros(input_ids.shape[0], input_ids.shape[1], 4, dtype=torch.long)

        with torch.no_grad():
            loss, logits = model(input_ids, attention_mask, token_type_ids, bbox)
            predictions = model.get_predictions(logits, attention_mask)

        id2label = {0: 'O', 1: 'B-COMPANY', 2: 'I-COMPANY', 3: 'B-DATE',
                    4: 'I-DATE', 5: 'B-ADDRESS', 6: 'I-ADDRESS', 7: 'B-TOTAL', 8: 'I-TOTAL'}

        # Map predictions to the original word tokens
        word_ids = encoding.word_ids()
        pred_flat = predictions[0]

        # Get prediction for first subword of each word
        word_preds = {}
        for tok_idx, wid in enumerate(word_ids):
            if wid is not None and wid not in word_preds and tok_idx < len(pred_flat):
                word_preds[wid] = pred_flat[tok_idx]

        print("\nTest predictions:")
        for i, token in enumerate(sample_tokens):
            pid = word_preds.get(i, 0)
            label = id2label.get(pid, 'O')
            print(f"  {token:<12} -> {label}")

        entity_preds = [id2label.get(word_preds.get(i, 0), 'O') for i in range(len(sample_tokens))]
        non_o = [l for l in entity_preds if l != 'O']
        if non_o:
            print(f"\n🎉 Model predicts {len(non_o)} entity labels!")
        else:
            print(f"\n⚠️  Model predicts only O labels – may need more training")
    else:
        print("❌ No checkpoint found for testing")

except Exception as e:
    print(f"❌ Error testing model: {e}")
    import traceback
    traceback.print_exc()

## 9. Download

After training completes:

1. **Best checkpoint**: `/kaggle/working/checkpoints/best_model.pt` (~400 MB)
2. **Final checkpoint**: `/kaggle/working/checkpoints/final_model.pt`
3. **Training curves**: `/kaggle/working/checkpoints/training_curves.png`
4. **Training log**: `/kaggle/working/checkpoints/training_log.json`

Use the Kaggle "Output" tab to download these files.

---

**Expected training time:** ~2–4 hours on Kaggle T4 GPU